In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import ast
import os
import glob
import numpy as np
import faiss
import torch
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from collections import Counter
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import gc
import gcsfs
import fsspec
import time
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
from queue import Queue, Empty
import startup
from datetime import datetime # Adding for session ID
from google.cloud import storage, bigquery
from pathlib import Path
from startup import load_indices

start_time = time.time()

# ------------------------------------------------------------------
# MOD 4 - FAISS index cache configuration (GCS -> local, concurrency-safe)
# ------------------------------------------------------------------

FAISS_REMOTE_INDEX_URI = (
    "gs://capstone_patent_bucket/faiss_index/localsnippetUSPatent/"
    "ivfflatnprobe16NLIST1024.faiss"
)

FAISS_CACHE_DIR = os.path.join("faiss_index_cache")
FAISS_LOCAL_INDEX_PATH = os.path.join(
    FAISS_CACHE_DIR,
    "localsnippetUSPatent_ivfflatnprobe16NLIST1024.faiss"
)
os.makedirs(FAISS_CACHE_DIR, exist_ok=True)


def ensure_faiss_index_local() -> str:
    """
    Ensure the FAISS index file exists locally.
    If missing, download it once from GCS into FAISS_LOCAL_INDEX_PATH.
    Uses a .lock file so parallel runs don't step on each other.
    Returns the local path to the index.
    """
    local_path = FAISS_LOCAL_INDEX_PATH
    lock_path = local_path + ".lock"

    # Fast path: index already cached locally
    if os.path.exists(local_path):
        return local_path

    # Try to become the "builder" by creating the lock file atomically
    we_are_builder = False
    try:
        fd = os.open(lock_path, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
        os.close(fd)
        we_are_builder = True
    except FileExistsError:
        we_are_builder = False

    if we_are_builder:
        # We are responsible for downloading the index from GCS
        try:
            print(f"[FAISS] Local index not found. Downloading from {FAISS_REMOTE_INDEX_URI} ...")
            res = subprocess.run(
                ["gsutil", "cp", FAISS_REMOTE_INDEX_URI, local_path],
                capture_output=True,
                text=True,
            )
            if res.returncode != 0:
                raise RuntimeError(f"gsutil cp failed:\n{res.stderr}")

            if not os.path.exists(local_path):
                raise FileNotFoundError(
                    f"Download reported success but {local_path} was not created."
                )

            print(f"[FAISS] Downloaded FAISS index to {local_path}")
        finally:
            # Always remove the lock, even if download failed
            try:
                os.remove(lock_path)
            except FileNotFoundError:
                pass
    else:
        # Another process is downloading; wait for it to finish or timeout
        print("[FAISS] Waiting for another process to finish downloading the FAISS index...")
        for _ in range(60):  # wait up to ~120 seconds
            if not os.path.exists(lock_path):
                break
            time.sleep(2)

    # After either we built it or someone else did, the file should exist
    if not os.path.exists(local_path):
        raise FileNotFoundError(
            f"FAISS index not found at {local_path} after download/wait."
        )

    return local_path

# ========== INDICES LOADING ==========
# Load FAISS indices from GCS (call explicitly to ensure they're available)
if startup.index is None:
    print("Loading FAISS indices from GCS...")
    startup.load_indices()
else:
    print("Indices already loaded (from startup.py)")

index = startup.index
metadata_offsets = startup.metadata_offsets
offset = startup.offset
patent_id_map = startup.patent_id_map

# Initialize cache with values from startup
_index_cache = {'index': index} if index else {}
_metadata_offsets_cache = {
    'offsets': metadata_offsets, 
    'offset': offset
} if metadata_offsets else {}
_patent_id_map_cache = {'patent_id_map': patent_id_map} if patent_id_map else {}

# Validate
assert index is not None, "FAISS index failed to load"
assert metadata_offsets is not None, "Metadata offsets failed to load"
assert patent_id_map is not None, "Patent ID map failed to load"
print(f"✓ FAISS Index: {index.ntotal} vectors")
print(f"✓ Metadata Offsets: {offset:,} records")
print(f"✓ Patent ID Map: {len(patent_id_map):,} IDs\n")

start_time = time.time()

# paths
parquet_path = "gs://capstone_patent_bucket/us_patent_parquets"
embeddings_path = "gs://capstone_patent_bucket/USPatentLocalEmbeddings"
FAISS_INDEX_PATH = "gs://capstone_patent_bucket/faiss_index/localsnippetUSPatent/ivfflatnprobe16NLIST1024.faiss"
METADATA_OFFSETS_PATH = "gs://capstone_patent_bucket/faiss_index/localsnippetUSPatent/ivfflat_metadata_offsets.csv" # New cache file
# NEW: Path for the Patent ID to Global Index Map (CHANGED TO .parquet for speed)
PATENT_ID_MAP_PATH = "gs://capstone_patent_bucket/faiss_index/localsnippetUSPatent/patent_id_map.parquet" 

#### --- MOD 1 - parameters: UI will inject these: --- ###
required = ["PATENT_TO_SEARCH", "SEARCH_TYPE", "TOP_K_RESULTS", "SESSION_PREFIX"]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(
        f"Missing required parameters: {missing}. "
        "Run via the Streamlit UI (click 'Run Search') or define these variables before executing the notebook."
    )




#NEW PATENT NUMBER NORMALIZATION BLOCK
import re

def normalize_patent_number(pn):
    if not pn or not isinstance(pn, str):
        raise ValueError("Patent number is empty or invalid")

    s = pn.strip().upper()
    s = re.sub(r"[^A-Z0-9]", "", s)  # remove spaces, dashes, punctuation

    COUNTRY_CODES = ["US","EP","WO","CN","KR","JP","DE","FR","GB","AU","CA","BR","IN","RU"]

    cc = None
    for code in sorted(COUNTRY_CODES, key=len, reverse=True):
        if s.startswith(code):
            cc = code
            rest = s[len(code):]
            break

    if cc is None:
        raise ValueError(f"Unknown country code in '{pn}'")

    # number (4–12 digits) + kind code (at least 1 letter, up to 3 chars)
    m = re.match(r"^(\d{4,12})([A-Z][A-Z0-9]{0,2})$", rest)
    if not m:
        raise ValueError(f"Unable to parse number + kind code in '{pn}'")

    number = m.group(1)
    kind = m.group(2)

    return f"{cc}-{number}-{kind}"


# Normalize the user-supplied patent number
try:
    CODE_DASHED = normalize_patent_number(PATENT_TO_SEARCH)
except Exception as e:
    raise ValueError(f"PATENT_TO_SEARCH is invalid: {e}")

# Compact version used elsewhere
CODE_COMPACT = CODE_DASHED.replace("-", "")
# -------------------------------------------------------------------


# SEARCH_TYPE validation (unchanged)
SEARCH_TYPE = str(SEARCH_TYPE).lower()
if SEARCH_TYPE not in {"publication", "application", "auto"}:
    raise ValueError("SEARCH_TYPE must be one of: 'publication', 'application', 'auto'.")

# TOP K validation
TOPK_MAX_CAP = 100_000

if isinstance(TOP_K_RESULTS, str) and TOP_K_RESULTS.strip().lower() == "all":
    TOP_K_RESULTS = -1

try:
    TOP_K_RESULTS = int(TOP_K_RESULTS)
except Exception as e:
    raise ValueError("TOP_K_RESULTS must be an integer or 'ALL'.") from e

if TOP_K_RESULTS <= 0:
    TOP_K_RESULTS = TOPK_MAX_CAP

#### --- END OF MOD 1 --- ###

### ----MOD 3: session-aware outputs under ./outputs ----
# Defaults when running notebook by itself (papermill overrides these)
# PATENT_TO_SEARCH = "US-10581355-B1"
# SEARCH_TYPE = "auto"
# TOP_K_RESULTS = 10

# SESSION_PREFIX is injected directly by the Streamlit UI via papermill.
# When running this notebook manually (outside the UI), we fall back to a simple local prefix.
try:
    SESSION_PREFIX
except NameError:
    # Fallback for interactive / debugging use only:
    try:
        CODE_DASHED = normalize_patent_number(PATENT_TO_SEARCH)
        CODE_COMPACT = CODE_DASHED.replace("-", "")
        SESSION_PREFIX = f"local_{CODE_COMPACT}__"
    except Exception:
        SESSION_PREFIX = "local__"


# Flat outputs layout under ./outputs (no per-session subfolder)
OUTPUTS_DIR   = "outputs"
PRIMARY_DIR   = os.path.join(OUTPUTS_DIR, "primary")
SECONDARY_DIR = os.path.join(OUTPUTS_DIR, "secondary")
HTML_DIR      = os.path.join(OUTPUTS_DIR, "html")
VISUALS_DIR_1 = os.path.join(OUTPUTS_DIR, "visuals_level_1")
VISUALS_DIR_2 = os.path.join(OUTPUTS_DIR, "visuals_level_2")

os.makedirs(OUTPUTS_DIR, exist_ok=True)
for d in (PRIMARY_DIR, SECONDARY_DIR, HTML_DIR, VISUALS_DIR_1, VISUALS_DIR_2):
    os.makedirs(d, exist_ok=True)

PRIMARY_CSV = os.path.join(
    PRIMARY_DIR,
    f"{SESSION_PREFIX}rag_search_from_patent_abstract_results_with_connections.csv",
)
SECONDARY_CSV = os.path.join(
    SECONDARY_DIR,
    f"{SESSION_PREFIX}rag_search_second_connections.csv",
)
OUTPUT_HTML = os.path.join(
    HTML_DIR,
    f"{SESSION_PREFIX}patent_connection_network.html",
)


# Visualization paths
# Level 1 visuals
IMG_SIM_DIST_1 = os.path.join(
    VISUALS_DIR_1,
    f"{SESSION_PREFIX}similarity_distribution.png",
)
IMG_WORDCLOUD_1 = os.path.join(
    VISUALS_DIR_1,
    f"{SESSION_PREFIX}keyword_wordcloud.png",
)
IMG_TOP_ASSIG_1 = os.path.join(
    VISUALS_DIR_1,
    f"{SESSION_PREFIX}top_10_assignees_L1.png",
)
IMG_TOP_INVEN_1 = os.path.join(
    VISUALS_DIR_1,
    f"{SESSION_PREFIX}top_10_inventors_L1.png",
)

# Level 2 visuals (only what actually exists in the notebook)
IMG_TOP_ASSIG_2 = os.path.join(
    VISUALS_DIR_2,
    f"{SESSION_PREFIX}top_10_assignees_L2.png",
)
IMG_TOP_INVEN_2 = os.path.join(
    VISUALS_DIR_2,
    f"{SESSION_PREFIX}top_10_inventors_L2.png",
)
IMG_CPC_CODE_2 = os.path.join(
    VISUALS_DIR_2,
    f"{SESSION_PREFIX}cpc_code_by_letter.png",
)

# --- File Cleanup for New Run (only this session's files) ---
#print(f"Cleaning up previous output files for session: {SESSION_ID}")

# for filename in [PRIMARY_CSV, SECONDARY_CSV, OUTPUT_HTML]:
#     if os.path.exists(filename):
#         try:
#             os.remove(filename)
#             print(f"  - Removed file: {filename}")
#         except Exception as e:
#             print(f"  - WARNING: Could not remove {filename}: {e}")

# print("Cleaning up previous visual PNGs for this session...")
# if os.path.exists(VISUALS_DIR_1):
#     for fname in os.listdir(VISUALS_DIR_1):
#         if fname.startswith(SESSION_PREFIX) and fname.lower().endswith(".png"):
#             fpath = os.path.join(VISUALS_DIR_1, fname)
#             try:
#                 os.remove(fpath)
#                 print(f"  - Removed visual: {fpath}")
#             except Exception as e:
#                 print(f"  - WARNING: Could not remove visual {fpath}: {e}")

# print("Cleanup complete.")
### --- End of MOD 3 ---

# --- use these defaults only when running notebook by itself ---
#PATENT_TO_SEARCH = "US-10581355-B1"
#SEARCH_TYPE = "auto"
#TOP_K_RESULTS = 10
#PRIMARY_CSV = "rag_search_from_patent_abstract_results_with_connections.csv"
#SECONDARY_CSV = "rag_search_second_connections.csv"
#OUTPUT_HTML = "patent_connection_network.html"

#use the available number of CPUs for the thread pool
MAX_WORKERS = 16

# GPU and sentence transformer model
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "all-MiniLM-L6-v2"

# FAISS Constants for IndexIVFFlat
NLIST = 1024
TRAINING_SIZE = 1_000_000
M_CODES = 32
NBits = 8

# Instantiate the GCS FileSystem once
fs = gcsfs.GCSFileSystem()


# FAISS SAVE/LOAD UTILITIES

def save_faiss_index(index, index_path, fs):
    """Saves the FAISS index to GCS using fsspec/gcsfs."""
    print(f"\n--- Saving FAISS index to {index_path} ---")
    try:
        local_path = "temp_index.faiss"
        faiss.write_index(index, local_path)

        # Upload to GCS
        with open(local_path, 'rb') as f:
            fs.pipe(index_path, f.read())

        os.remove(local_path)
        print("FAISS index saved successfully.")
    except Exception as e:
        print(f"ERROR saving FAISS index: {e}")

def load_faiss_index(index_path, fs):
    """
    Loads the FAISS IndexIVFFlat via a local cache, backed by GCS.

    - Uses ensure_faiss_index_local() to download the index once from GCS
      into FAISS_LOCAL_INDEX_PATH in a concurrency-safe way.
    - Then reads the FAISS index from that local cached file.
    """
    print(f"\n--- Attempting to load FAISS index (cached) for {index_path} ---")

    try:
        # This will download from GCS to FAISS_LOCAL_INDEX_PATH once,
        # using a .lock file to avoid race conditions between parallel runs.
        local_path = ensure_faiss_index_local()

        # Load the FAISS index from the local cached file
        index = faiss.read_index(local_path)
        print(f"FAISS index loaded successfully from cache: {local_path} ({index.ntotal} vectors).")
        return index

    except Exception as e:
        print(f"ERROR loading FAISS index from cache: {e}. Will rebuild.")
        return None

def load_or_build_faiss_index(embedding_files, dimension, index_type):
    """Load existing FAISS index or build and save a new one."""
    
    # Check if already in memory
    if 'index' in _index_cache:
        print(f"{index_type} index already loaded in memory")
        return _index_cache['index']

    index_path = f"{FAISS_INDEX_PATH}"
    
    # Try to load existing index
    print(f"\n--- Attempting to load {index_type} index ---")
    index = load_faiss_index(index_path, fs)
    
    if index is not None:
        print(f"{index_type} index loaded successfully with {index.ntotal} vectors")
        return index
    
    # Index doesn't exist, build a new one
    print(f"--- Building new {index_type} FAISS index ---")
    
    quantizer = faiss.IndexFlatIP(dimension)
    index = faiss.IndexIVFPQ(quantizer, dimension, NLIST, M_CODES, NBits, faiss.METRIC_INNER_PRODUCT)
    index.is_trained = False
    
    # Build training set
    training_vectors = []
    vector_count = 0
    
    print(f"Loading up to {TRAINING_SIZE:,} vectors for {index_type} training...")
    for f_path in embedding_files:
        try:
            with fs.open(f_path, 'rb') as f:
                batch_data = torch.load(f, map_location="cpu")
                if isinstance(batch_data, dict):
                    emb_tensor = batch_data['embeddings']
                else:
                    emb_tensor = batch_data
                    
                emb_np = emb_tensor.numpy().astype("float32")
            
            if vector_count + emb_np.shape[0] > TRAINING_SIZE:
                slice_size = TRAINING_SIZE - vector_count
                if slice_size > 0:
                    training_vectors.append(emb_np[:slice_size])
                break
            else:
                training_vectors.append(emb_np)
                vector_count += emb_np.shape[0]
            
            del emb_tensor, emb_np
            gc.collect()
        except Exception as e:
            print(f"Skipping {f_path}: {e}")
    
    if not training_vectors:
        raise RuntimeError(f"Failed to load any embeddings for {index_type}")
    
    # Train index
    training_set = np.concatenate(training_vectors, axis=0)
    print(f"Training set: {training_set.shape[0]:,} vectors")
    faiss.normalize_L2(training_set)
    index.train(training_set)
    print(f"{index_type} index trained")
    del training_set
    gc.collect()
    
    # Add all embeddings
    print(f"Adding all {index_type} embeddings to index...")
    for f_path in tqdm(embedding_files, desc=f"Adding {index_type}"):
        try:
            with fs.open(f_path, 'rb') as f:
                batch_data = torch.load(f, map_location="cpu")
                if isinstance(batch_data, dict):
                    emb = batch_data['embeddings']
                else:
                    emb = batch_data
                    
                emb_np = emb.numpy().astype("float32")
            
            faiss.normalize_L2(emb_np)
            index.add(emb_np)
            del emb_np
            gc.collect()
        except Exception as e:
            print(f"Skipping {f_path}: {e}")
    
    print(f"{index_type} index built with {index.ntotal} vectors")
    
    # Save for future use
    save_faiss_index(index, index_path, fs)
    _index_cache['index'] = index
    
    return index

# METADATA OFFSET UTILITIES

def calculate_and_save_offsets(gcs_parquet_path, gcs_offset_path, fs):
    """Calculates all metadata offsets by reading every Parquet file header and saves the result to GCS."""
    print("\n--- Calculating all metadata offsets (This may take several minutes) ---")
    
    parquet_files_list = sorted(fs.glob(gcs_parquet_path.replace("gs://", "") + "/*.parquet"))
    parquet_files_rag = [f'gs://{f}' for f in parquet_files_list]
    
    metadata_offsets = []
    offset = 0

    for pf in tqdm(parquet_files_rag, desc="Processing Parquet files for row counts"):
        try:
            # Pass fs directly for reading Parquet
            df_count = pd.read_parquet(pf, columns=["publication_number"], filesystem=fs)
            rows = len(df_count)
            del df_count
            # Store (file_path, start_index, end_index)
            metadata_offsets.append((pf, offset, offset + rows))
            offset += rows
        except Exception as e:
            print(f"Error reading {pf} for row count: {e}")
            
    # Save the calculated offsets to GCS as a CSV
    offsets_df = pd.DataFrame(metadata_offsets, columns=['file_path', 'start', 'end'])
    

    offsets_df.to_csv(gcs_offset_path, index=False, storage_options={'fs': fs})
    
    print(f"Offsets saved to {gcs_offset_path}. Total records: {offset:,}")
    return metadata_offsets, offset

def load_or_calculate_offsets(gcs_parquet_path, gcs_offset_path, fs):
    """Tries to load pre-calculated offsets from GCS; calculates and saves them if not found."""
    
    # Check if already in memory
    if 'offsets' in _metadata_offsets_cache:
        print("Metadata offsets already loaded in memory")
        return _metadata_offsets_cache['offsets'], _metadata_offsets_cache['offset']    
        
    print(f"\n--- Attempting to load metadata offsets from {gcs_offset_path} ---")
    
    try:
        offsets_df = pd.read_csv(gcs_offset_path, storage_options={'fs': fs})
        metadata_offsets = list(offsets_df.itertuples(index=False, name=None))
        offset = metadata_offsets[-1][2] if metadata_offsets else 0
        
        print(f"Offsets loaded successfully from GCS. Total records: {offset:,}")
        _metadata_offsets_cache['offset'] = offset
        return metadata_offsets, offset
        
    except Exception as e:       
        path_parts = gcs_offset_path.replace("gs://", "").split("/", 1)
        if len(path_parts) < 2 or not fs.exists(path_parts[0] + '/' + path_parts[1]):
            print(f"Offset file not found. Initiating calculation.")
        else:
            print(f"Error loading offset file: {e}. Initiating calculation as a fallback.")

        # Fallback to calculation
        metadata_offsets, offset = calculate_and_save_offsets(gcs_parquet_path, gcs_offset_path, fs)
        # Cache it
        _metadata_offsets_cache['offsets'] = metadata_offsets
        _metadata_offsets_cache['offset'] = offset
        return metadata_offsets, offset

# PATENT ID MAP UTILITIES (MODIFIED CODE FOR PARQUET)

def calculate_and_save_patent_id_map(gcs_parquet_path, gcs_map_path, fs):
    """Calculates map from patent_number to global FAISS index ID and saves to GCS."""
    print("--- Calculating Patent ID to Global Index Map (This may take a while) ---")
    
    parquet_files_list = sorted(fs.glob(gcs_parquet_path.replace("gs://", "") + "/*.parquet"))
    parquet_files_rag = [f'gs://{f}' for f in parquet_files_list]
    
    patent_id_to_global_index = {}
    global_offset = 0

    for pf in tqdm(parquet_files_rag, desc="Processing Parquet files for ID map"):
        try:
            # Load only the ID columns
            df_ids = pd.read_parquet(pf, columns=["publication_number", "application_number"], filesystem=fs)
            
            # Create a local ID map for this file
            for i, row in df_ids.iterrows():
                # Map Publication Number
                pub_num = str(row["publication_number"]).strip()
                if pub_num:
                    patent_id_to_global_index[pub_num] = global_offset + i
                
                # Map Application Number (as a fallback or alternative search key)
                app_num = str(row["application_number"]).strip()
                if app_num and app_num not in patent_id_to_global_index:
                    patent_id_to_global_index[app_num] = global_offset + i
            
            global_offset += len(df_ids)
            del df_ids
            gc.collect()

        except Exception as e:
            print(f"Error reading {pf} for ID map: {e}")
            
    # Save the map to GCS as a DataFrame/Parquet (MODIFIED: to_parquet)
    map_df = pd.DataFrame(
        list(patent_id_to_global_index.items()), 
        columns=['patent_id', 'global_index']
    )

    map_df.to_parquet(gcs_map_path, index=False, storage_options={'fs': fs})
    
    print(f"Patent ID map saved to {gcs_map_path}. Total IDs: {len(patent_id_to_global_index):,}")
    return patent_id_to_global_index

def load_or_calculate_patent_id_map(gcs_parquet_path, gcs_map_path, fs):
    """Tries to load pre-calculated patent ID map from GCS; calculates and saves them if not found."""

    # Check if already in memory
    if 'patent_id_map' in _patent_id_map_cache:
        print("Patent ID map already loaded in memory")
        return _patent_id_map_cache['patent_id_map']
    
    print(f"\n--- Attempting to load Patent ID Map from {gcs_map_path} ---")
    
    try:
        # MODIFIED: read_parquet for much faster loading
        map_df = pd.read_parquet(gcs_map_path, storage_options={'fs': fs})
        # Convert DataFrame to a dictionary (Series) for fast in-memory lookup
        patent_id_to_global_index = pd.Series(
            map_df.global_index.values, 
            index=map_df.patent_id
        ).to_dict()
        
        print(f"Patent ID Map loaded successfully from GCS. Total IDs: {len(patent_id_to_global_index):,}")
        # Cache it
        _patent_id_map_cache['patent_id_map'] = patent_id_to_global_index
        return patent_id_to_global_index
        
    except Exception as e:
        path_parts = gcs_map_path.replace("gs://", "").split("/", 1)
        if len(path_parts) < 2 or not fs.exists(path_parts[0] + '/' + path_parts[1]):
            print(f"Patent ID Map file not found. Initiating calculation.")
        else:
            print(f"Error loading Patent ID Map file: {e}. Initiating calculation as a fallback.")

        # Fallback to calculation
        patent_id_to_global_index = calculate_and_save_patent_id_map(gcs_parquet_path, gcs_map_path, fs)
        # Cache it
        _patent_id_map_cache['patent_id_map'] = patent_id_to_global_index
        return patent_id_to_global_index
    

# METADATA UTILITIES

def extract_text(cell, lang="en"):
    """Extract text from localized fields, preferring the target language but falling back if unavailable."""
    if cell is None:
        return None
    if isinstance(cell, np.ndarray):
        try:
            cell = cell.tolist()
        except Exception:
            pass
    if pd.isna(cell) if not isinstance(cell, (list, dict)) else False:
        return None
    try:
        if isinstance(cell, str):
            try:
                data = ast.literal_eval(cell)
            except (ValueError, SyntaxError):
                data = cell
        else:
            data = cell
        if isinstance(data, list):
            for item in data:
                if isinstance(item, dict) and item.get("language") == lang:
                    return item.get("text")
            for item in data:
                if isinstance(item, dict) and "text" in item:
                    return item["text"]
        elif isinstance(data, dict):
            return data.get("text")
        elif isinstance(data, str):
            return data
    except Exception:
        return None
    return None


def get_patent_info(patent_number, search_type="auto"):
    """
    MODIFIED: Retrieves patent details using the in-memory patent ID map (O(1) lookup).
    Uses the existing get_metadata_row(global_idx) to fetch data once the index is found.
    """
    patent_number = str(patent_number).strip()

    # 1. Look up the Global Index ID using the pre-calculated map
    global_idx = patent_id_to_global_index.get(patent_number)

    if global_idx is None:
        print(f"\nPatent {patent_number} not found in the ID map.")
        # Return consistent structure when not found
        return {
            "publication_number": None,
            "application_number": None,
            "title": None,
            "abstract": None,
        }

    print(f"\nFound patent {patent_number} at global index ID: {global_idx}. Retrieving metadata...")

    # 2. Retrieve the full row using the existing, optimized get_metadata_row
    match = get_metadata_row(global_idx)

    if match is None:
        print(f"Error: Could not retrieve metadata for index {global_idx}.")
        return {
            "publication_number": None,
            "application_number": None,
            "title": None,
            "abstract": None,
        }

    # 3. Extract text and return info (same as old logic)
    title, abstract = extract_text_from_result_row(match)

    print("\nPatent Info Found:")
    print(f"Publication Number: {match.get('publication_number', 'N/A')}")
    print(f"Title: {title if title else 'N/A'}")
    print(f"Abstract: {abstract[:100] + '...' if abstract and len(abstract) > 100 else (abstract if abstract else 'N/A')}")

    return {
        "publication_number": match.get("publication_number"),
        "application_number": match.get("application_number"),
        "title": title,
        "abstract": abstract
    }

# RAG

# load Sentence Transformer Model
embed_model = SentenceTransformer(model_name, device=device)

# Full List of Embedding Files
embedding_files_list = sorted(fs.glob(os.path.join(embeddings_path.replace("gs://", ""), "*.pt")))
embedding_files_for_faiss = [f'gs://{f}' for f in embedding_files_list]

if not embedding_files_for_faiss:
    raise FileNotFoundError(f"No embedding files found in {embeddings_path}")
print(f"Found {len(embedding_files_for_faiss)} embedding files.")


# FAISS INDEX LOAD OR BUILD LOGIC
# Set embedding dimension (MiniLM produces 384-dimensional embeddings)
dimension = 384

# Use your existing load_or_build_faiss_index function
# This function already handles: load if exists, build and save if not
index = load_or_build_faiss_index(
    embedding_files=embedding_files_for_faiss,
    dimension=dimension,
    index_type="patent_idmap2"
)

if index is None:
    raise RuntimeError("Failed to load or build FAISS index")

print(f"FAISS index ready with {index.ntotal} vectors")
print(f"MARKER A (FAISS Load Complete): {time.time() - start_time:.4f} seconds elapsed.")

# ==============================================================================
# METADATA OFFSET LOAD/BUILD
# ==============================================================================

# This replaces the old slow block with a fast load or a slow build-and-save.
metadata_offsets, offset = load_or_calculate_offsets(parquet_path, METADATA_OFFSETS_PATH, fs)

print(f"MARKER B (Metadata Offsets Ready): {time.time() - start_time:.4f} seconds elapsed.") # Time marker B

# ==============================================================================
# PATENT ID MAP LOAD/BUILD (NEW STEP, NOW USING PARQUET)
# ==============================================================================
patent_id_to_global_index = load_or_calculate_patent_id_map(parquet_path, PATENT_ID_MAP_PATH, fs)

print(f"MARKER C' (Patent ID Map Ready): {time.time() - start_time:.4f} seconds elapsed.")

# We need the metadata offset list/tuple to define get_metadata_row correctly.
# Redefine get_metadata_row now that metadata_offsets is defined.
def get_metadata_row(global_idx):
    for pf, start, end in metadata_offsets:
        if start <= global_idx < end:
            local_idx = global_idx - start
            try:
                # Read the specific Parquet file using the stored path
                df = pd.read_parquet(pf, filesystem=fs)
                return df.iloc[local_idx]
            except Exception as e:
                print(f"Error reading {pf}: {e}")
                return None
    return None

# The rest of the functions remain the same...

def parse_connection_records(records):
    connections = set()
    if records is None:
        return []

    if isinstance(records, str):
        try:
            records = ast.literal_eval(records)
        except (ValueError, SyntaxError):
            return [p.strip() for p in records.split(',') if p.strip()]

    if not isinstance(records, (list, np.ndarray)):
        return [str(records).strip()] if records else []

    for record in records:
        if isinstance(record, dict):
            num = record.get('application_number') or record.get('publication_number')
            if num and str(num).strip():
                connections.add(str(num).strip())
        elif isinstance(record, str) and record.strip():
            connections.add(record.strip())

    return sorted(list(connections))


def get_connections(row):
    parents_raw = row.get("parent")
    parent_list = parse_connection_records(parents_raw)

    children_raw = row.get("child")
    child_list = parse_connection_records(children_raw)

    connections = {
        "parent_connections": parent_list,
        "child_connections": child_list,
    }
    return connections


def try_parse(obj):
    """Safely parse JSON-like strings (dict/list) into Python objects."""
    if isinstance(obj, str):
        try:
            parsed = ast.literal_eval(obj)
            return parsed
        except Exception:
            return obj
    return obj

def extract_top_keywords(text, top_n=5):
    """Extract top keywords from text using simple frequency-based approach."""
    if not isinstance(text, str) or not text.strip():
        return []

    # clean and tokenize
    tokens = re.findall(r"\b[a-zA-Z]{3,}\b", text.lower())
    filtered = [t for t in tokens if t not in ENGLISH_STOP_WORDS]

    if not filtered:
        return []

    # count most common words
    counts = Counter(filtered)
    top_words = [word for word, _ in counts.most_common(top_n)]
    return top_words


def extract_text_from_result_row(row):
    """
    Attempts to robustly extract English title and abstract from the metadata row
    using the known 'title_en' and 'abstract_en' columns, with fallback logic.
    """
    try:
        title = None
        abstract = None

        # 1. Try to retrieve directly from known English columns
        title = str(row.get("title_en")).strip() if row.get("title_en") is not None else None
        abstract = str(row.get("abstract_en")).strip() if row.get("abstract_en") is not None else None

        # Clean up generic None/nan strings
        if title and title.lower() in ['none', 'nan', 'n/a', '']:
            title = None
        if abstract and abstract.lower() in ['none', 'nan', 'n/a', '']:
            abstract = None

        # 2. Fallback using the complex extract_text logic for other columns (in case data is inconsistent)
        if not title:
            title = extract_text(row.get("title"))  # extract_text(row.get("title_localized")) retrieves BOOLEAN

        if not abstract:
            abstract = extract_text(row.get("abstract"))  # extract_text(row.get("abstract_localized")) retrieves BOOLEAN

        return title, abstract
    except Exception as e:
        print(f"Error extracting text: {e}")
        return None, None

# ---------------------------------------------
#       PARALLEL FIRST-LEVEL SEARCH MOD
# ---------------------------------------------

def search_worker(search_queue, patent_found):
    """Threaded first-level patent search"""
    
    while not patent_found.is_set():  # Exit if patent found
        try:
            # Get the next file to search (blocks if queue empty)
            job = search_queue.get(timeout=60) # 60 second timeout

            if job is None: # Sentinel value to stop worker
                print("[SEARCH-WORKER] First-level search complete. Shutting down...")
                break

            batch_files = job['batch_files']
            batch_num = job['batch_num']
            worker_id = job['worker_id']
            patent_abstract = job['patent_abstract']

            print(f"[SEARCH-WORKER {worker_id}] Processing batch {batch_num}")

            # Search this batch using the new batch-specific function
            results = rag_search_faiss_batch(
                patent_abstract, 
                TOP_K_RESULTS, 
                PRIMARY_CSV,
                batch_files=batch_files,
                worker_id=worker_id
            )

            # Check if patent was found in results
            if results and len(results) > 0:
                found_patents = [r.get('Publication_Number') for r in results]
                if PATENT_TO_SEARCH in found_patents:
                    print(f"[SEARCH-WORKER {worker_id}] Found {PATENT_TO_SEARCH}!")
                    patent_found.set()

        except Empty:
            continue
        except Exception as e:
            print(f"[SEARCH-WORKER] Error: {e}")
            
# ---------------------------------------------
#      END PARALLEL FIRST-LEVEL SEARCH MOD
# ---------------------------------------------

def rag_search_faiss(query_text, top_k, output_csv, batch_indices=None, is_second_level=False, query_row=None, connection_type=None, connection_pub_num=None):
    # Ensure index exists before setting nprobe
    if index is None:
        print("ERROR: FAISS index is not loaded/built. Cannot perform search.")
        return

    if not query_text or str(query_text).strip() == "":
        print("ERROR: Query text (abstract) is empty. Cannot perform RAG search.")
        return

    # QUALITY FIX: Increase nprobe
    index.nprobe = 16

    # QUALITY FIX: Normalization
    query_emb = embed_model.encode(query_text, convert_to_numpy=True).reshape(1, -1)

    distances, indices = index.search(query_emb, top_k)

    results = []

    # Print less verbose for parallel searches
    print(f"Searching for connection {connection_pub_num} ({connection_type})...")

    for rank, (idx, dist) in enumerate(zip(indices[0], distances[0]), start=1):
        # Skip if filtering to a batch and idx not in batch
        if batch_indices is not None and idx not in batch_indices:
            continue
        
        # idx is the global ID
        if idx < 0: # Handle invalid indices
            continue

        row = get_metadata_row(idx)
        if row is None:
            if not is_second_level:
                print(f"Rank {rank}: Metadata missing for index {idx}")
            continue

        # QUALITY FIX: Metadata Retrieval
        # Use the robust helper function
        title, abstract = extract_text_from_result_row(row)

        # extract metadata fields
        inventors = try_parse(row.get("inventor_harmonized", row.get("inventor", "")))
        assignees = try_parse(row.get("assignee_harmonized", row.get("assignee", "")))


        # extract CPC codes
        cpc_codes = []
        cpc_raw = row.get("cpc", row.get("fi", None))

        if isinstance(cpc_raw, str):
            try:
                parsed = ast.literal_eval(cpc_raw)
            except Exception:
                parsed = cpc_raw
        elif isinstance(cpc_raw, np.ndarray):
            parsed = cpc_raw.tolist()
        else:
            parsed = cpc_raw

        if isinstance(parsed, dict):
            parsed = [parsed]
        elif isinstance(parsed, str):
            parsed = [parsed]

        if isinstance(parsed, list):
            for item in parsed:
                if isinstance(item, dict):
                    code = item.get("symbol") or item.get("code") or ""
                    if code:
                        cpc_codes.append(code)
                elif isinstance(item, str):
                    if re.match(r"^[A-HY]\d{2}[A-Z]\d+/?\d*", item): # CPC pattern
                        cpc_codes.append(item)

        cpc_codes = sorted(set(cpc_codes))


        # extract top 5 keywords from abstract
        top_keywords = extract_top_keywords(abstract, top_n=5)

        # parent/child connections
        try:
            connections = get_connections(row)
        except Exception:
            connections = {"parent_connections": [], "child_connections": []}

        # console output
        print(f"\nRank {rank}: (Score {dist:.4f})")
        print(f"Publication Number: {row.get('publication_number', 'N/A')}")
        print(f"Title: {title if title else 'None'}")
        print(f"CPC Codes: {cpc_codes if cpc_codes else 'N/A'}")
        print(f"Top Keywords: {top_keywords if top_keywords else 'N/A'}")
        print(f"Parent Connections: {connections.get('parent_connections', [])}")
        print(f"Child Connections: {connections.get('child_connections', [])}")
        print(f"Similarity Score: {dist:.4f}")
        print(f"Abstract: {abstract[:100] + '...' if abstract and len(abstract) > 100 else (abstract if abstract else 'N/A')}")
        print("-" * 100)

        # save results
        result = {
            "Rank": rank,
            "Similarity_Score": float(dist),
            "Publication_Number": row.get("publication_number", ""),
            "Application_Number": row.get("application_number", ""),
            "Title": str(title),
            "Inventors": str(inventors),
            "Assignees": str(assignees),
            "Parent_Connections": str(connections.get("parent_connections", [])),
            "Child_Connections": str(connections.get("child_connections", [])),
            "CPC_Codes": str(cpc_codes),
            "Top_Keywords": str(top_keywords),
            "Abstract": str(abstract),
        }

        # Add connection-specific columns if this is a second-level search
        if is_second_level:
            result["SecondConnectionTo"] = connection_type
            if connection_type == "Parent":
                result["SourceParentPublication"] = connection_pub_num
                result["SourceChildPublication"] = np.nan
            else:
                result["SourceChildPublication"] = connection_pub_num
                result["SourceParentPublication"] = np.nan

        # Append to the list
        results.append(result)

    # write results to CSV (kept local)
    if not is_second_level:
        pd.DataFrame(results).to_csv(output_csv, index=False, encoding="utf-8")
        print(f"\nSaved top {top_k} results to: {output_csv}")
    
    print(f"MARKER: {time.time() - start_time:.4f} seconds elapsed.")
    return results

def rag_search_faiss_batch(query_text, top_k, output_csv, batch_files, worker_id):
    """
    Wrapper that searches only within a specific batch of files.
    """
    
    # Filter metadata_offsets to only include files in this batch
    batch_offsets = [mo for mo in metadata_offsets if mo[0] in batch_files]
    
    if not batch_offsets:
        print(f"[SEARCH-WORKER {worker_id}] No offsets found for batch files")
        return []
    
    # Create set of valid indices for this batch
    batch_indices = set()
    for pf, start, end in batch_offsets:
        batch_indices.update(range(start, end))
    
    print(f"[SEARCH-WORKER {worker_id}] Searching {len(batch_indices)} records in batch")

    # Call rag_search_faiss with batch filter
    results = rag_search_faiss(query_text, top_k, output_csv, batch_indices=batch_indices, is_second_level=False)
    
    return results

# Function to handle a single connection search
def search_single_connection(pub, connection_type, top_k):
    """Retrieves patent info and runs RAG search for a single connection."""
    patent_info = get_patent_info(pub, SEARCH_TYPE)
    if not patent_info or not patent_info.get("abstract"):
        print(f"Skipping {connection_type} Connection {pub}: No abstract found.")
        return None

    # NOTE: output_csv is now temporary/unused in this function
    results = rag_search_faiss(
        patent_info["abstract"], 
        top_k, 
        output_csv="", 
        is_second_level=True, 
        connection_type=connection_type, 
        connection_pub_num=pub
    )
    
    # Convert results (list of dicts) to DataFrame for consistency
    if results:
        temp_df = pd.DataFrame(results)
        return temp_df
    return None

# query execution
if __name__ == "__main__":
    print(f"Starting Patent Abstract RAG Process for Patent: {PATENT_TO_SEARCH}")

    patent_info = get_patent_info(PATENT_TO_SEARCH, SEARCH_TYPE)

    print(f"MARKER D (Patent Info Retrieved): {time.time() - start_time:.4f} seconds elapsed.") # Time marker D
    
    # Create and start search worker thread
    patent_found = threading.Event()
    search_queue = Queue()
    search_worker_thread = threading.Thread(
        target=search_worker,
        args=(search_queue, patent_found),
        daemon=False
    )
    search_worker_thread.start()

    num_workers = 4  # Adjust based on testing; try 4-8 for 16 vCPUs
    worker_threads = [search_worker_thread]
    print("[MAIN] Search worker thread started")
    
    for i in range(num_workers - 1):
        worker = threading.Thread(
            target=search_worker,
            args=(search_queue, patent_found),
            daemon=False
        )
        worker.start()
        worker_threads.append(worker)
        print(f"[MAIN] Search worker thread {i+2} started")

    # Get files with gs:// prefix
    parquet_files = sorted([f'gs://{f}' for f in fs.glob(parquet_path.replace("gs://", "") + "/*.parquet")])

    BATCH_SIZE = 20

    # Process in batches
    total_batches = (len(parquet_files) + BATCH_SIZE - 1) // BATCH_SIZE
    start_batch = 0

    try:
        if patent_info and patent_info.get("abstract"):    
            patent_abstract = patent_info["abstract"]
            print("\n--- Patent Abstract Retrieved. Starting RAG Search ---")

            # Distribute parquet FILES (not batches) among workers
            batches_per_worker = (len(parquet_files) + num_workers - 1) // num_workers
    
            for worker_id in range(num_workers):
                worker_start_batch = worker_id * batches_per_worker
                worker_end_batch = min((worker_id + 1) * batches_per_worker, total_batches)
    
                # Each worker gets its own subset of parquet files
                worker_files = parquet_files[worker_start_batch:worker_end_batch]
                
                # Split worker's files into batches of BATCH_SIZE
                for batch_num, i in enumerate(range(0, len(worker_files), BATCH_SIZE)):
                    batch = worker_files[i:i+BATCH_SIZE]
                    
                    search_queue.put({
                        'batch_files': batch,
                        'batch_num': batch_num,
                        'worker_id': worker_id,
                        'patent_abstract': patent_abstract
                    })
        else:
            print("\nFATAL: Could not retrieve valid abstract. RAG search aborted.")
    finally:
        # Signal all workers to stop
        for _ in range(num_workers):
            search_queue.put(None)
        
        # Wait for all workers to finish
        for worker in worker_threads:
            worker.join()
    
    # NOW run rag_search_faiss() once with the patent abstract
    if patent_info and patent_info.get("abstract"):
        results = rag_search_faiss(patent_abstract, TOP_K_RESULTS, PRIMARY_CSV, is_second_level=False)
            
    print("\nProcess complete.")
    print(f"MARKER: {time.time() - start_time:.4f} seconds elapsed.")

#=================================================================================================================================================
## Second Connections - Parallel Execution

if __name__ == "__main__":

    for f in glob.glob(f"{SESSION_PREFIX}_temp_*.csv"): os.remove(f)  # At start of second-level search
    def safe_parse_list(val):
        """Safely parse connection lists from CSV."""
        if isinstance(val, list):
            return val
        if not isinstance(val, str) or not val.strip():
            return []
        try:
            parsed = ast.literal_eval(val)
            if isinstance(parsed, list):
                return parsed
            return []
        except Exception:
            return []
    def run_second_connection_search(base_csv, top_k=TOP_K_RESULTS, original_title=None):
        if not os.path.exists(base_csv):
            print(f"File not found: {base_csv}")
            return

        df = pd.read_csv(base_csv)
        all_parent_pubs = set()
        all_child_pubs = set()

        # Gather parent/child connections
        for _, row in df.iterrows():
            parents = safe_parse_list(row.get("Parent_Connections", "[]"))
            children = safe_parse_list(row.get("Child_Connections", "[]"))
            all_parent_pubs.update(parents)
            all_child_pubs.update(children)

        # Filter out the original patent from the connections to avoid searching it again
        all_parent_pubs.discard(PATENT_TO_SEARCH)
        all_child_pubs.discard(PATENT_TO_SEARCH)

        print(f"\nFound {len(all_parent_pubs)} unique parent connections.")
        print(f"Found {len(all_child_pubs)} unique child connections.")

        total_connections = len(all_parent_pubs) + len(all_child_pubs)
        
        if total_connections == 0:
            print("No connections to search.")
            return

        all_connections = []
        for pub in all_parent_pubs:
            all_connections.append((pub, "Parent", top_k))
        for pub in all_child_pubs:
            all_connections.append((pub, "Child", top_k))

        results_all = []
        
        # --- PARALLEL EXECUTION BLOCK ---
        start_parallel_time = time.time()
        print(f"Starting parallel search for {total_connections} connections using {MAX_WORKERS} workers...")
        
        # Use ThreadPoolExecutor for I/O bound tasks like GCS file access and network calls for the embedding model
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            # Submits all the connection searches
            future_to_connection = {
                executor.submit(search_single_connection, pub, conn_type, k): (pub, conn_type)
                for pub, conn_type, k in all_connections
            }
            
            # Wait for results as they complete
            for future in tqdm(as_completed(future_to_connection), total=len(future_to_connection), desc="Parallel Connection Searches"):
                pub, conn_type = future_to_connection[future]
                try:
                    df_result = future.result()
                    if df_result is not None:
                        results_all.append(df_result)
                except Exception as exc:
                    print(f'{pub} generated an exception: {exc}')

        print(f"MARKER E (Parallel Search Complete): {time.time() - start_time:.4f} seconds elapsed.") # Time marker E

        # --- END OF PARALLEL EXECUTION BLOCK ---

        # Combine all results
        if results_all:
            combined_df = pd.concat(results_all, ignore_index=True)

            # Insert original patent number at top
            first_row_data = {col: "" for col in combined_df.columns}
            first_row_data["Publication_Number"] = PATENT_TO_SEARCH
            first_row_data["Title"] = original_title or 'Original Patent'
            first_row_data["SecondConnectionTo"] = "Original"

            first_row = pd.DataFrame([first_row_data])
            combined_df = pd.concat([first_row, combined_df], ignore_index=True)

            combined_df.to_csv(SECONDARY_CSV, index=False, encoding="utf-8")
            print(f"\nSecond-level connection search complete.")
            print(f"Saved combined results to: {SECONDARY_CSV}")

        else:
            print("\nNo valid second-level results found.")


    # Run second connection expansion
    if patent_info is not None:
        run_second_connection_search(PRIMARY_CSV, original_title=patent_info.get('title'))
    else:
        print("Skipping second-level search: Patent not found in index")

    # Cache the original patent title to avoid re-searching later
    original_patent_title = patent_info.get('title') if patent_info else 'Original Patent'

#==============================================================================================================================================

# patent connection network
# --- MOD2-re-added for testing ---
if __name__ == "__main__":
    import networkx as nx
    from pyvis.network import Network

    # config (re-defined locally for standalone execution)
    MAIN_CSV = PRIMARY_CSV
    SECOND_CSV = SECONDARY_CSV

# --- End of MOD2 ---

def safe_parse_list(val):
    if isinstance(val, list):
        return val
    if not isinstance(val, str) or not val.strip():
        return []
    try:
        parsed = ast.literal_eval(val)
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []

# --- NEW: guard against missing CSVs (no abstract / no 2nd-level results) ---
import os
import pandas as pd

if not os.path.exists(MAIN_CSV):
    print(f"\n[INFO] Skipping network graph: main CSV not found: {MAIN_CSV}")
else:
    df_main = pd.read_csv(MAIN_CSV)

    if os.path.exists(SECOND_CSV):
        df_second = pd.read_csv(SECOND_CSV)
    else:
        print(f"\n[INFO] No second-level CSV at {SECOND_CSV}. Building network from primary results only.")
        # Empty frame with same columns so concat() still works
        df_second = pd.DataFrame(columns=df_main.columns)

    # ------------------------------------------------------------------
    # Build a lookup table from publication/application numbers to titles
    # ------------------------------------------------------------------
    title_lookup = {}

    combined = pd.concat([df_main, df_second], ignore_index=True)
    for _, r in combined.iterrows():
        pub = str(r.get("Publication_Number", "")).strip()
        app = str(r.get("Application_Number", "")).strip()
        title = str(r.get("Title", "")).strip()

        if title and title.lower() != "nan":
            # Map by publication number
            if pub:
                title_lookup[pub] = title
            # Also map by application number if present and not already set
            if app and app not in title_lookup:
                title_lookup[app] = title

    # Convenience set of all publication numbers from both levels
    all_pub_numbers = set(df_main["Publication_Number"]).union(df_second["Publication_Number"])

    # For any connections that are in the CSVs but not yet in title_lookup,
    # pull their titles from df_main.
    for _, r in df_main.iterrows():
        for conn_list in ["Parent_Connections", "Child_Connections"]:
            for conn in safe_parse_list(r.get(conn_list, "[]")):
                conn = str(conn).strip()
                if conn and conn not in title_lookup and conn in all_pub_numbers:
                    vals = df_main.loc[df_main["Publication_Number"] == conn, "Title"].values
                    if len(vals) > 0:
                        title_lookup[conn] = str(vals[0]).strip()

    # ------------------------------------------------------------------
    # Build the NetworkX graph
    # ------------------------------------------------------------------
    import networkx as nx
    from pyvis.network import Network

    G = nx.DiGraph()

    # Level 1 (primary) parent/child relationships
    for _, row in df_main.iterrows():
        pub = str(row["Publication_Number"]).strip()
        parents = safe_parse_list(row.get("Parent_Connections", "[]"))
        children = safe_parse_list(row.get("Child_Connections", "[]"))

        for p in parents:
            G.add_edge(str(p).strip(), pub, relation="parent_to_child")
        for c in children:
            G.add_edge(pub, str(c).strip(), relation="parent_to_child")

    # Level 2 (second-level) relationships using source columns
    for _, row in df_second.iterrows():
        pub = str(row.get("Publication_Number", "")).strip()
        src_parent = row.get("SourceParentPublication", None)
        src_child = row.get("SourceChildPublication", None)

        if src_parent and not pd.isna(src_parent):
            G.add_edge(str(src_parent).strip(), pub, relation="second_parent_to_child")
        if src_child and not pd.isna(src_child):
            G.add_edge(str(src_child).strip(), pub, relation="second_child_to_child")

    # Query patent → result edges
    orig = str(PATENT_TO_SEARCH).strip()
    if orig not in G.nodes:
        G.add_node(orig)

    for _, row in df_main.iterrows():
        pub = str(row["Publication_Number"]).strip()
        if pub and pub != orig:
            G.add_edge(orig, pub, relation="query_to_result")

    # Ensure we have a title for the original patent
    if orig not in title_lookup:
        title_lookup[orig] = f"Patent {orig} (Title Not Found in CSV)"

    # ------------------------------------------------------------------
    # Build the PyVis network graph
    # ------------------------------------------------------------------
    net = Network(
        height="850px",
        width="100%",
        bgcolor="#111111",
        font_color="white",
        directed=True,
    )
    net.force_atlas_2based(gravity=-50)

    # Nodes
    for node in G.nodes():
        label = str(node)
        title = title_lookup.get(label, f"Patent {label} (Title Not Found)")
        if label == orig:
            # Center star for the query/original patent
            net.add_node(
                label,
                label=label,
                title=title,
                color="#FFD700",
                size=40,
                shape="star",
                physics=False,
                x=0,
                y=0,
            )
        else:
            net.add_node(label, label=label, title=title, color="#00ccff")

    # Edges with relation-based colors and arrows
    for source, target, data in G.edges(data=True):
        rel = data.get("relation", "")
        color = {
            "parent_to_child": "#00ccff",
            "second_parent_to_child": "#ff9900",
            "second_child_to_child": "#ff3366",
            "query_to_result": "#66ff66",
        }.get(rel, "#cccccc")

        net.add_edge(
            source,
            target,
            color=color,
            arrows="to",
            title=rel,
            width=3,
        )

    net.set_options("""
    var options = {
      "edges": {
        "arrows": { "to": { "enabled": true, "scaleFactor": 1.0 }},
        "smooth": { "enabled": false },
        "color": { "inherit": false }
      },
      "nodes": {
        "shape": "dot",
        "scaling": { "min": 8, "max": 50 }
      },
      "physics": {
        "barnesHut": {
          "gravitationalConstant": -20000,
          "springLength": 200,
          "centralGravity": 0.1
        }
      }
    }
    """)

    # Ensure HTML directory exists and write network HTML
    os.makedirs(os.path.dirname(OUTPUT_HTML), exist_ok=True)
    net.write_html(OUTPUT_HTML)

    # Inject legend into HTML, but only if the file exists
    if os.path.exists(OUTPUT_HTML):
        with open(OUTPUT_HTML, "r", encoding="utf-8") as f:
            html = f.read()

        legend_html = """
    <div id="legend" style="
        position: fixed;
        bottom: 20px;
        right: 20px;
        background: rgba(20,20,20,0.92);
        color: white;
        border: 2px solid #777;
        border-radius: 10px;
        padding: 18px 20px;
        font-size: 16px;
        z-index: 9999;
        width: 280px;
        box-shadow: 0px 0px 10px rgba(0,0,0,0.4);
    ">
      <b style="font-size: 20px">Legend</b><br><br>

      <span style="color:#00ccff; font-size:20px">■</span>
        Parent → Child<br><br>

      <span style="color:#ff9900; font-size:20px">■</span>
        2nd Parent → Child<br><br>

      <span style="color:#ff3366; font-size:20px">■</span>
        2nd Child → Child<br><br>

      <span style="color:#66ff66; font-size:20px">■</span>
        Query → Result<br><br>

      <span style="color:#FFD700; font-size:22px">★</span>
        Query Patent (Original)
    </div>
    """

        final_html = html.replace("</body>", f"{legend_html}</body>")

        with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
            f.write(final_html)
    else:
        print(f"[INFO] Skipping legend injection: HTML file not found ({OUTPUT_HTML})")

    print(f"\nNetwork visualization saved: {OUTPUT_HTML}")


#==============================================================================================================================================

# visual plots

import matplotlib.pyplot as plt
import ast
from collections import Counter
import os
import pandas as pd
import re
import json
from wordcloud import WordCloud 

def visualize_results(csv_file, save_dir, plot_similarity=False, plot_wordcloud=False, plot_assignees=False, plot_inventors=False, plot_cpc_by_letter=False):
    """
    Generates and saves various plots based on the content of a CSV file, 
    controlled by boolean flags for granular control over output per data level.
    
    NOTE: This function uses column names matching the uploaded CSV files:
    - Similarity Score is read from 'Similarity_Score'.
    - Title/Abstract for Wordcloud are read from 'Title' and 'Abstract'.
    - CPC Codes are read from 'CPC_Codes'.

    Args:
        csv_file (str): Path to the input CSV file.
        save_dir (str): Directory to save the generated visual plots.
        plot_similarity (bool): Plot the cosine similarity distribution.
        plot_wordcloud (bool): Plot the keyword wordcloud.
        plot_assignees (bool): Plot the top 10 assignees by frequency.
        plot_inventors (bool): Plot the top 10 inventors by frequency.
        plot_cpc_by_letter (bool): Plot the CPC code frequency by main section letter.
    """
    print(f"\n--- Generating visualizations for {os.path.basename(csv_file)} (Saving to {save_dir}) ---")

    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    try:
        # The engine='python' is added to handle potential mixed dtypes warnings gracefully
        df = pd.read_csv(csv_file, engine='python')
    except FileNotFoundError:
        print(f"Error: CSV file not found at {csv_file}")
        return

    # Determine the connection level for the title
    if 'level_1' in save_dir:
        level_label = "Level 1 Connections"
    elif 'level_2' in save_dir:
        level_label = "Level 2 and Level 3 Connections"
    else:
        level_label = os.path.basename(csv_file) # Fallback to filename

        # Decide filename suffix based on level, for L1 vs L2 plots
    if "level_1" in save_dir:
        level_suffix = "L1"
    elif "level_2" in save_dir:
        level_suffix = "L2"
    else:
        level_suffix = ""  # fallback, no suffix

    # Helper function to safely parse and extract names/codes using aggressive string/regex matching
    def safe_parse_and_flatten(series, column_name):
        flattened_list = []
        
        # Regex to match the content of the 'name' field, accommodating single or double quotes
        # It looks for: 'name': ' (or ") and captures the text until the closing ' (or ")
        name_regex = re.compile(r"'name'\s*:\s*['\"]([^'\"]+)['\"]")

        for item in series.astype(str).dropna().unique():
            raw_item = item.strip()
            if not raw_item or raw_item.lower() in ['nan', 'none', '[]']:
                continue
            
            if column_name in ['Inventors', 'Assignees']:
                # For Inventors/Assignees, use regex to extract names from the list-of-dict string format
                names = name_regex.findall(raw_item)
                flattened_list.extend([name.strip() for name in names])
            
            elif column_name == 'CPC_Codes':
                # For CPC Codes, try to parse it as a simple list of strings
                try:
                    # Fallback to ast.literal_eval for simple lists (like CPC_Codes or keywords)
                    parsed_list = ast.literal_eval(raw_item)
                    if isinstance(parsed_list, list):
                        flattened_list.extend([str(x).strip().strip("'\"") for x in parsed_list])
                    else:
                        flattened_list.append(str(parsed_list).strip().strip("'\""))
                except (ValueError, SyntaxError):
                    # If parsing fails, treat the whole item as a single CPC code or keyword (unlikely for CPC)
                    flattened_list.append(raw_item)

        # Final clean-up: filter out empty strings and common placeholders
        return [item for item in flattened_list if item and item.lower() not in ['nan', 'none']]


    # 1. Similarity Score Distribution Plot (Level 1 Only) - USING 'Similarity_Score'
    if plot_similarity and 'Similarity_Score' in df.columns and len(df) > 0:
        plt.figure(figsize=(10, 6))
        data_to_plot = df['Similarity_Score'].dropna()
        if len(data_to_plot) > 0:
            plt.hist(data_to_plot, bins=20, edgecolor='black', alpha=0.7)
            # Use the new level_label in the title
            plt.title(f"Similarity Score Distribution ({level_label})")
            plt.xlabel("Cosine Similarity Score")
            plt.ylabel("Frequency")
            plt.grid(axis='y', alpha=0.5)
            plt.tight_layout()
            plt.savefig(os.path.join(save_dir, f"{SESSION_PREFIX}similarity_score_distribution.png"))
        else:
            print("Skipping Similarity Plot: No valid 'Similarity_Score' data.")
        plt.close()
    elif plot_similarity:
        print("Skipping Similarity Plot: 'Similarity_Score' column missing or no data.")

    # 2. Keyword Wordcloud (Level 1 or 2) - USING 'Title' and 'Abstract'
    if plot_wordcloud and 'Title' in df.columns and 'Abstract' in df.columns:
        print("Generating keyword wordcloud...")
        try:
            # Define a robust set of stop words including common English stop words
            standard_english_stop_words = set([
                'a', 'about', 'above', 'after', 'again', 'against', 'all', 'am', 'an', 'and', 'any', 'are', "aren't", 'as', 'at', 
                'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 
                'can', "can't", 'cannot', 'could', "couldn't", 'did', "didn't", 'do', 'does', "doesn't", 'doing', "don't", 
                'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', "hadn't", 'has', "hasn't", 'have', "haven't", 
                'having', 'he', "he'd", "he'll", "he's", 'her', 'here', "here's", 'hers', 'herself', 'him', 'himself', 'his', 
                'how', "how's", 'i', "i'd", "i'll", "i'm", "i've", 'if', 'in', 'into', 'is', "isn't", 'it', "it's", 'its', 
                'itself', 'just', 'me', 'more', 'most', "mustn't", 'my', 'myself', 'no', 'nor', 'not', 'of', 'off', 'on', 
                'once', 'only', 'or', 'other', 'ought', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 'same', "shan't", 
                'she', "she'd", "she'll", "she's", 'should', "shouldn't", 'so', 'some', 'such', 'than', 'that', "that's", 
                'the', 'their', 'theirs', 'them', 'themselves', 'then', 'there', "there's", 'these', 'they', "they'd", 
                "they'll", "they're", "they've", 'this', 'those', 'through', 'to', 'too', 'under', 'until', 'up', 'very', 
                'was', "wasn't", 'we', "we'd", "we'll", "we're", "we've", 'were', "weren't", 'what', "what's", 'when', 
                "when's", 'where', "where's", 'which', 'while', 'who', "who's", 'whom', 'why', "why's", 'with', "won't", 
                'would', "wouldn't", 'you', "you'd", "you'll", "you're", "you've", 'your', 'yours', 'yourself', 'yourselves',
            ])
            
            # Custom patent/data stop words
            patent_stop_words = set(['nan', 'none', 'said', 'may', 'can', 'one', 'two', 'system', 'method', 'apparatus', 'device', 'include', 'including'])
            
            stop_words = standard_english_stop_words.union(patent_stop_words)

            # Simple keyword extraction (titles and abstracts)
            text_data = ' '.join(df['Title'].astype(str) + ' ' + df['Abstract'].astype(str))
            words = text_data.lower().split()
            words = [re.sub(r'[^a-z]', '', word) for word in words]
            
            # Apply the combined stop word filter
            words = [word for word in words if word and len(word) > 2 and word not in stop_words] 
            freq_dict = Counter(words)

            if freq_dict:
                wc = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(freq_dict)
                plt.figure(figsize=(10, 6))
                plt.imshow(wc, interpolation="bilinear")
                plt.axis("off")
                # Use the new level_label in the title
                plt.title(f"Keyword Wordcloud (Titles + Abstracts) ({level_label})")
                plt.tight_layout()
                plt.savefig(os.path.join(save_dir, f"{SESSION_PREFIX}keyword_wordcloud.png"))
            else:
                print("Skipping Wordcloud: No significant keywords found.")
        except Exception as e:
            print(f"Error generating Wordcloud: {e}")
        plt.close()
    elif plot_wordcloud:
        print("Skipping Wordcloud: 'Title' or 'Abstract' column missing.")

    # 3. Top 10 Assignees Plot (Level 1 & 2)
    if plot_assignees and 'Assignees' in df.columns and len(df) > 0:
        print("Generating Top 10 Assignees plot...")
        # Get clean assignee names using the updated, aggressive parsing
        assignees = safe_parse_and_flatten(df['Assignees'], 'Assignees')
        
        if assignees:
            top_assignees = Counter(assignees).most_common(10)
            names, counts = zip(*top_assignees)
            
            # --- START: Name Simplification/Truncation for Assignees ---
            def truncate_assignee_name(name, max_len=30, max_words=4): 
                """Aggressively truncates long names for better plot readability, preferring words over raw characters."""
                words = name.split()
                if len(name) > max_len and len(words) > max_words:
                    # Take the first max_words, join them, and truncate the result
                    shortened_name = ' '.join(words[:max_words])
                    if len(shortened_name) > max_len:
                        return shortened_name[:max_len-3].strip() + '...'
                    return shortened_name + '...'
                elif len(name) > max_len:
                    return name[:max_len-3].strip() + '...'
                return name
                
            simplified_assignee_names = [truncate_assignee_name(name) for name in names]
            # --- END: Name Simplification/Truncation ---

            # --- START: Readability Fix for Assignees (Increased width, use x-axis for counts) ---
            plt.figure(figsize=(15, 9))
            # Plot the bars
            plt.barh(simplified_assignee_names[::-1], counts[::-1], color='skyblue') 
            
            # Set the x-axis to use only integer ticks
            plt.gca().xaxis.set_major_locator(plt.MaxNLocator(integer=True))
            
            # Set y-ticks and restore x-ticks/label
            plt.yticks(fontsize=8)
            plt.xlabel("Frequency (Number of Patents)") 
            
            # --- END: Readability Fix ---
            
            # Use the new level_label in the title
            plt.title(f"Top 10 Assignees by Frequency ({level_label})")
            plt.ylabel("Assignee Name")
            plt.tight_layout()
            plt.savefig(
                os.path.join(
                    save_dir,
                    f"{SESSION_PREFIX}top_10_assignees{('_' + level_suffix) if level_suffix else ''}.png"
                )
            )

        else:
            print("Skipping Assignees Plot: No valid assignee data found.")
        plt.close()
    elif plot_assignees:
        print("Skipping Assignees Plot: 'Assignees' column missing.")


    # 4. Top 10 Inventors Plot (Level 1 & 2)
    if plot_inventors and 'Inventors' in df.columns and len(df) > 0:
        print("Generating Top 10 Inventors plot...")
        # Get clean inventor names using the updated, aggressive parsing
        inventors = safe_parse_and_flatten(df['Inventors'], 'Inventors')

        if inventors:
            top_inventors = Counter(inventors).most_common(10)
            names, counts = zip(*top_inventors)

            # --- START: Name Simplification for Inventors (Last Name Only) ---
            def simplify_inventor_name(full_name):
                """Simplifies the inventor name for visualization to just the Last Name (first word)."""
                if not full_name:
                    return ""
                # Assume the last name is the first word in the patent data format (e.g., KRASOWSKI MICHAEL -> KRASOWSKI)
                return full_name.split(' ')[0].strip()
            
            simplified_names = [simplify_inventor_name(name) for name in names]
            # --- END: Name Simplification ---
            
            # --- START: Readability Fix for Inventors (Increased width, use x-axis for counts) ---
            plt.figure(figsize=(15, 9))
            # Plot the bars
            plt.barh(simplified_names[::-1], counts[::-1], color='lightcoral') 
            
            # Set the x-axis to use only integer ticks
            plt.gca().xaxis.set_major_locator(plt.MaxNLocator(integer=True))
            
            # Set y-ticks and restore x-ticks/label
            plt.yticks(fontsize=10) 
            plt.xlabel("Frequency (Number of Patents)") 
            
            # --- END: Readability Fix ---
            
            # Use the new level_label in the title
            plt.title(f"Top 10 Inventors by Frequency ({level_label})")
            plt.ylabel("Inventor Name (Last Name Only)")
            plt.tight_layout()
            plt.savefig(
                os.path.join(
                    save_dir,
                    f"{SESSION_PREFIX}top_10_inventors{('_' + level_suffix) if level_suffix else ''}.png"
                )
            )

        else:
            print("Skipping Inventors Plot: No valid inventor data found.")
        plt.close()
    elif plot_inventors:
        print("Skipping Inventors Plot: 'Inventors' column missing.")

    # 5. CPC Code Plot (By Letter) (Level 2 Only) - USING 'CPC_Codes'
    if plot_cpc_by_letter and 'CPC_Codes' in df.columns and len(df) > 0:
        print("Generating CPC Code (By Letter) plot...")
        # Note: CPC_Codes still uses ast.literal_eval since it's a simple list of strings
        cpc_codes = safe_parse_and_flatten(df['CPC_Codes'], 'CPC_Codes')
        
        cpc_letters = []
        for code in cpc_codes:
            # Extract the main section letter (first character)
            if code and code[0].isalpha():
                cpc_letters.append(code[0].upper())
        
        if cpc_letters:
            letter_counts = Counter(cpc_letters)
            # Sort alphabetically by letter for standard CPC section order
            sorted_counts = sorted(letter_counts.items(), key=lambda item: item[0])
            
            letters, counts = zip(*sorted_counts)
            
            plt.figure(figsize=(10, 6))
            plt.bar(letters, counts, color='lightgreen')
            # Use the new level_label in the title
            plt.title(f"CPC Code Frequency by Main Section Letter ({level_label})")
            plt.xlabel("CPC Section Letter")
            plt.ylabel("Frequency of Patents")
            plt.grid(axis='y', alpha=0.5)
            plt.tight_layout()
            plt.savefig(os.path.join(save_dir, f"{SESSION_PREFIX}cpc_code_by_letter.png"))
        else:
            print("Skipping CPC Plot: No valid CPC code data found.")
        plt.close()
    elif plot_cpc_by_letter:
        print("Skipping CPC Plot: 'CPC_Codes' column missing.")

    print(f"--- Visualizations for {level_label} complete. ---")


# The final execution block to run visualizations based on the new requirements

# Run visualizations for both levels with specific plot requirements
try:
    # Level 1: Similarity Score, Assignees, Inventors, Wordcloud
    # NOTE: PRIMARY_CSV corresponds to the Level 1 file: rag_search_from_patent_abstract_results_with_connections (8).csv
    visualize_results(
        PRIMARY_CSV, 
        save_dir=str(VISUALS_DIR_1),
        plot_similarity=True,   # Keep Similarity Score (L1)
        plot_wordcloud=True,    # Wordcloud is on L1
        plot_assignees=True,    # Keep Assignees (L1)
        plot_inventors=True     # Keep Inventors (L1)
    )
    
    # Level 2: Assignees, Inventors, CPC Code by Letter (Wordcloud removed)
    # NOTE: SECONDARY_CSV corresponds to the Level 2 file: rag_search_second_connections.csv
    visualize_results(
        SECONDARY_CSV, 
        save_dir=str(VISUALS_DIR_2),
        plot_similarity=False,
        plot_wordcloud=False,   # Wordcloud is off L2
        plot_assignees=True,    # Keep Assignees (L2)
        plot_inventors=True,    # Keep Inventors (L2)
        plot_cpc_by_letter=True # Keep CPC Code by Letter (L2)
    )
    print("\nVisual plots generation complete.")
except NameError:
    print("\nERROR: Configuration constants (PRIMARY_CSV, SECONDARY_CSV) are missing from the global scope. Cannot run visualization.")
except Exception as e:
    print(f"\nAn unexpected error occurred during visualization execution: {e}")
print(f"MARKER F (Complete): {time.time() - start_time:.4f} seconds elapsed.")